# 05a · Modèle 1 : corrections et modèle final (durée de livraison et promesse par quantile)
**Projet : E-Commerce Intelligence Platform** · CRISP-DM, phase 5 (Evaluation), avant le déploiement

**Pourquoi ce notebook ?** Le rapport d'évaluation a montré trois fragilités sur le modèle 1 :
1. les **baselines** (médianes) étaient calculées sur le train seul, alors que le modèle final était réentraîné sur train + valid : la comparaison n'était pas équitable ;
2. la **dérive temporelle** (livraisons plus rapides en 2018) rend le test unique peu fiable ;
3. `orders_last_7d` et `purchase_month` pourraient servir de **proxys du temps** plutôt que d'expliquer les délais.

**Ce que fait ce notebook**
1. **Évaluation glissante par mois** : chaque mois est prédit par des modèles entraînés sur *tous les mois précédents*, ce qui reproduit un usage réel avec réentraînement mensuel.
2. **Baselines équitables** : médiane globale, médiane par état, médiane par état sur les 3 derniers mois, promesse d'Olist, toutes recalculées avec le même historique que le modèle.
3. **Ablation** : modèle avec toutes les variables, sans proxys du temps, sans proxys ni promesse, avec pondération par la récence.
4. **Simulateur de promesse** évalué lui aussi en glissant (taux de retard et durée promise, par mois).
5. **Modèle final** entraîné sur toutes les données et **fichiers de déploiement** (modèles, prédictions, métriques).

> Données : dataset Kaggle `e-commerce-dataset-model-1`. Durée indicative : 10 à 25 minutes. Les fichiers sont écrits dans `/kaggle/working/deploy/`.

## 1. Imports et paramètres

In [ ]:
import os, glob, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

SEED = 42
FAST = os.environ.get("FAST", "0") == "1"
OUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working/deploy")
os.makedirs(OUT_DIR, exist_ok=True)

PROXY = ["orders_last_7d", "purchase_month"]       # variables suspectées de servir de proxys du temps
HALF_LIFE = 90                                    # demi-vie (jours) de la pondération par la récence
N_EST = 100 if FAST else 400
ROLL_MIN_TRAIN = 200 if FAST else 5000            # taille minimale de l'historique pour évaluer un mois
Q_ROLL = [0.8, 0.9, 0.95]                         # quantiles évalués en glissant
Q_FINAL = [0.5, 0.7, 0.8, 0.85, 0.9, 0.95]        # quantiles du modèle final
FINAL_VARIANT = None                              # None : choix automatique ; sinon un nom de variante du tableau

# Chemins des jeux de données Kaggle (créés à partir de la sortie du notebook 03)
DATA_PATHS = {
    "model1_delivery_duration.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-1/model1_delivery_duration.csv",
    "model2_late_risk.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-2/model2_late_risk.csv",
    "model3_customer_segmentation.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-3/model3_customer_segmentation.csv",
}

def find_file(name):
    """DATA_DIR (si défini), puis le chemin Kaggle ci-dessus, puis une recherche dans /kaggle/input."""
    cands = []
    if os.environ.get("DATA_DIR"):
        cands.append(os.path.join(os.environ["DATA_DIR"], name))
    cands.append(DATA_PATHS.get(name, ""))
    cands += glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    for c in cands:
        if c and os.path.exists(c):
            return c
    raise FileNotFoundError(f"{name} introuvable : ajoute le dataset correspondant en entrée du notebook (Add Input).")

## 2. Chargement

In [ ]:
m1 = pd.read_csv(find_file("model1_delivery_duration.csv"), parse_dates=["order_purchase_timestamp"])

FEATURES_NUMERIC = [
    "nb_items", "multi_seller", "total_price", "total_freight", "total_weight_g", "total_volume_cm3",
    "weight_imputed", "freight_ratio", "limit_days", "distance_km", "distance_imputed", "same_state",
    "customer_lat", "customer_lng", "seller_lat", "seller_lng", "payment_installments",
    "purchase_month", "purchase_dow", "purchase_hour", "is_weekend", "is_black_friday_week", "orders_last_7d",
    "seller_n_past", "seller_mean_days", "seller_late_rate", "route_n_past", "route_mean_days", "route_late_rate",
    "log_total_price", "log_total_freight", "log_total_weight_g", "log_total_volume_cm3", "log_distance_km",
]
CATS = ["customer_state", "seller_state", "categorie_grouped", "payment_type"]
PROMISE = ["promised_days"]
FEATURES_CORE = FEATURES_NUMERIC + CATS

needed = FEATURES_CORE + PROMISE + ["delivery_days", "split", "order_id", "customer_state", "baseline_state_median_days"]
assert not [c for c in needed if c not in m1.columns], "Colonnes absentes"

full = m1.sort_values("order_purchase_timestamp").reset_index(drop=True)
full["month"] = full["order_purchase_timestamp"].dt.to_period("M").astype(str)
roll_months = sorted(full.loc[full["split"].isin(["valid", "test"]), "month"].unique())
test_months = sorted(full.loc[full["split"] == "test", "month"].unique())
print(f"{len(full):,} commandes | mois évalués en glissant : {roll_months}")
print("dont mois du bloc test :", test_months)

## 3. Fonctions communes

In [ ]:
def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {"MAE": mean_absolute_error(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "R2": r2_score(y, p), "MedAE": median_absolute_error(y, p), "Biais": float(np.mean(p - y))}

def make_converter(feats, ref):
    cats = [c for c in CATS if c in feats]
    levels = {c: sorted(ref[c].astype(str).unique()) for c in cats}
    def conv(df):
        X = df[feats].copy()
        for c in cats:
            X[c] = pd.Categorical(X[c].astype(str), categories=levels[c])
        return X
    conv.levels = levels
    return conv

LGB = dict(n_estimators=N_EST, learning_rate=0.05, num_leaves=31, min_child_samples=50, subsample=0.8, subsample_freq=1,
           colsample_bytree=0.7, reg_lambda=5.0, random_state=SEED, n_jobs=-1, verbose=-1)

def recency_weights(trn, start):
    age_days = (start - trn["order_purchase_timestamp"]).dt.total_seconds() / 86400
    return 0.5 ** (age_days / HALF_LIFE)

def fit_reg(trn, feats, weight, start, objective="regression_l1", alpha=None):
    conv = make_converter(feats, trn)
    p = {**LGB, "objective": objective}
    if alpha is not None:
        p["alpha"] = alpha
    m = lgb.LGBMRegressor(**p).fit(conv(trn), trn["delivery_days"].values,
                                   sample_weight=recency_weights(trn, start) if weight else None)
    return m, conv

FEATS_ALL = FEATURES_CORE + PROMISE
FEATS_NP = [f for f in FEATS_ALL if f not in PROXY]
FEATS_NPP = [f for f in FEATS_NP if f not in PROMISE]
VARIANTS = {
    "A · toutes variables": {"feats": FEATS_ALL, "weight": False},
    "B · sans proxys du temps": {"feats": FEATS_NP, "weight": False},
    "C · sans proxys ni promesse": {"feats": FEATS_NPP, "weight": False},
    "D · sans proxys, pondéré par la récence": {"feats": FEATS_NP, "weight": True},
    "E · toutes variables, pondéré par la récence": {"feats": FEATS_ALL, "weight": True},
}
print({k: len(v["feats"]) for k, v in VARIANTS.items()})

## 4. Évaluation glissante par mois
Pour chaque mois *m* (mars à août 2018) : entraînement sur **toutes les commandes antérieures au 1er du mois**, prédiction du mois. Les baselines utilisent exactement le même historique : médiane globale, médiane par état, et médiane par état des **3 derniers mois** (qui s'adapte à la dérive). La promesse d'Olist est connue à l'achat.

In [ ]:
BASE = ["Promesse d'Olist", "Médiane globale (historique)", "Médiane par état (historique)", "Médiane par état (3 derniers mois)"]
store = {n: {"err": [], "month": [], "y": []} for n in BASE + list(VARIANTS)}
roll_frames = []
t0 = time.time()
for mth in roll_months:
    start = pd.Period(mth, "M").start_time
    trn = full[full["order_purchase_timestamp"] < start]
    tst = full[full["month"] == mth]
    if len(trn) < ROLL_MIN_TRAIN or len(tst) == 0:
        print("  mois ignoré (historique insuffisant) :", mth); continue
    y = tst["delivery_days"].values
    gmed = trn["delivery_days"].median()
    smed = trn.groupby("customer_state")["delivery_days"].median()
    rmed = trn[trn["order_purchase_timestamp"] >= start - pd.DateOffset(months=3)].groupby("customer_state")["delivery_days"].median()
    s_hist = tst["customer_state"].map(smed).fillna(gmed).values
    preds = {"Promesse d'Olist": tst["promised_days"].values,
             "Médiane globale (historique)": np.full(len(tst), gmed),
             "Médiane par état (historique)": s_hist,
             "Médiane par état (3 derniers mois)": tst["customer_state"].map(rmed).fillna(pd.Series(s_hist, index=tst.index)).values}
    for vname, v in VARIANTS.items():
        m, conv = fit_reg(trn, v["feats"], v["weight"], start)
        preds[vname] = np.clip(m.predict(conv(tst)), 0, None)
    for n, p in preds.items():
        store[n]["err"].append(p - y); store[n]["month"].append(np.full(len(y), mth)); store[n]["y"].append(y)
    roll_frames.append(pd.DataFrame({"order_id": tst["order_id"].values, "month": mth, "split": tst["split"].values,
                                     "delivery_days": y, "promised_days": tst["promised_days"].values, **{f"pred::{n}": p for n, p in preds.items()}}))
    print(f"  {mth} : {len(trn):,} commandes d'entraînement, {len(tst):,} évaluées ({time.time() - t0:.0f} s)")
roll = pd.concat(roll_frames, ignore_index=True)

## 5. Résultats de l'évaluation glissante

In [ ]:
rows = []
for n, s in store.items():
    err, mon = np.concatenate(s["err"]), np.concatenate(s["month"])
    is_test = np.isin(mon, test_months)
    rows.append({"méthode": n, "type": "Baseline" if n in BASE else "LightGBM L1", "MAE (tous les mois)": np.abs(err).mean(),
                 "MAE (mois du test)": np.abs(err[is_test]).mean() if is_test.any() else np.nan,
                 "biais (tous)": err.mean(), "n": len(err)})
summary = pd.DataFrame(rows).sort_values("MAE (tous les mois)").reset_index(drop=True)
display(summary)

by_month = pd.DataFrame({n: pd.Series(np.abs(np.concatenate(s["err"]))).groupby(np.concatenate(s["month"])).mean() for n, s in store.items()})
display(by_month)

fig, ax = plt.subplots(1, 2, figsize=(18, 5))
for n in BASE:
    by_month[n].plot(ax=ax[0], ls="--", marker="x", label=n)
for n in VARIANTS:
    by_month[n].plot(ax=ax[0], marker="o", label=n)
ax[0].set_title("MAE par mois (modèles entraînés sur tous les mois précédents)"); ax[0].set_ylabel("jours"); ax[0].legend(fontsize=7)
sig = summary.set_index("méthode")["MAE (tous les mois)"].sort_values(ascending=False)
ax[1].barh(sig.index, sig.values, color=["tab:red" if i in BASE else "tab:blue" for i in sig.index])
ax[1].set_title("MAE sur l'ensemble des mois évalués")
plt.tight_layout(); plt.show()

## 6. Choix de la variante finale
Variante LightGBM de **MAE glissante minimale**. Écart avec la meilleure baseline équitable et avec la promesse d'Olist.

In [ ]:
ml = summary[summary["type"] != "Baseline"]
final_variant = FINAL_VARIANT if FINAL_VARIANT else ml.iloc[0]["méthode"]
best_base = summary[summary["type"] == "Baseline"].iloc[0]
fv = summary[summary["méthode"] == final_variant].iloc[0]
promise_mae = summary[summary["méthode"] == "Promesse d'Olist"].iloc[0]["MAE (tous les mois)"]
print(f"Variante retenue : {final_variant}")
print(f"  MAE glissante             : {fv['MAE (tous les mois)']:.3f} j")
print(f"  meilleure baseline        : {best_base['méthode']} = {best_base['MAE (tous les mois)']:.3f} j "
      f"-> gain {best_base['MAE (tous les mois)'] - fv['MAE (tous les mois)']:+.3f} j "
      f"({(best_base['MAE (tous les mois)'] - fv['MAE (tous les mois)']) / best_base['MAE (tous les mois)'] * 100:+.1f} %)")
print(f"  promesse d'Olist          : {promise_mae:.3f} j -> gain {promise_mae - fv['MAE (tous les mois)']:+.3f} j "
      f"({(promise_mae - fv['MAE (tous les mois)']) / promise_mae * 100:+.1f} %)")
a = summary[summary["méthode"] == "A · toutes variables"]["MAE (tous les mois)"].iloc[0]
b = summary[summary["méthode"] == "B · sans proxys du temps"]["MAE (tous les mois)"].iloc[0]
print(f"Effet du retrait de orders_last_7d et purchase_month : MAE {a:.3f} -> {b:.3f} j ({b - a:+.3f} j)")
chosen = VARIANTS[final_variant]

## 7. Comparaison statique équitable sur le bloc test
Modèle **et** baselines ajustés sur train + valid, évalués sur test : c'est le tableau qui remplace celui du rapport d'évaluation.

In [ ]:
trva = full[full["split"].isin(["train", "valid"])]
tst = full[full["split"] == "test"]
start_test = tst["order_purchase_timestamp"].min()
gmed = trva["delivery_days"].median()
smed = trva.groupby("customer_state")["delivery_days"].median()
rmed = trva[trva["order_purchase_timestamp"] >= start_test - pd.DateOffset(months=3)].groupby("customer_state")["delivery_days"].median()
s_hist = tst["customer_state"].map(smed).fillna(gmed).values
yte = tst["delivery_days"].values
m_static, conv_static = fit_reg(trva, chosen["feats"], chosen["weight"], start_test)
rows = [{"modèle": "Promesse d'Olist", **metrics(yte, tst["promised_days"])},
        {"modèle": "Médiane globale (train + valid)", **metrics(yte, np.full(len(tst), gmed))},
        {"modèle": "Médiane par état (train + valid)", **metrics(yte, s_hist)},
        {"modèle": "Médiane par état (3 derniers mois avant le test)",
         **metrics(yte, tst["customer_state"].map(rmed).fillna(pd.Series(s_hist, index=tst.index)).values)},
        {"modèle": f"LightGBM L1 : {final_variant}", **metrics(yte, np.clip(m_static.predict(conv_static(tst)), 0, None))}]
static_test = pd.DataFrame(rows).set_index("modèle")
display(static_test)
mm = static_test.iloc[-1]["MAE"]
for n in static_test.index[:-1]:
    print(f"Gain de MAE vs {n:52s}: {static_test.loc[n, 'MAE'] - mm:6.2f} j ({(static_test.loc[n, 'MAE'] - mm) / static_test.loc[n, 'MAE'] * 100:5.1f} %)")

## 8. Simulateur de promesse évalué en glissant
Pour chaque mois, des modèles LightGBM **quantiles** (entraînés sur les mois précédents) prédisent le quantile *q* de la durée. Promettre ce quantile devrait donner un taux de retard proche de `1 - q`. On compare, mois par mois, la **durée promise moyenne** et le **taux de retard** à ceux de la promesse d'Olist.

In [ ]:
q_store = {q: {"promise": [], "late": [], "n": []} for q in Q_ROLL}
o_store = {"promise": [], "late": [], "n": []}
q_frames = []
for mth in roll_months:
    start = pd.Period(mth, "M").start_time
    trn = full[full["order_purchase_timestamp"] < start]; tst_m = full[full["month"] == mth]
    if len(trn) < ROLL_MIN_TRAIN or len(tst_m) == 0:
        continue
    y = tst_m["delivery_days"].values
    o_store["promise"].append(tst_m["promised_days"].mean()); o_store["late"].append((y > tst_m["promised_days"].values).mean()); o_store["n"].append(len(y))
    cols = {"order_id": tst_m["order_id"].values}
    for q in Q_ROLL:
        mq, cq = fit_reg(trn, chosen["feats"], chosen["weight"], start, "quantile", q)
        pq = np.clip(mq.predict(cq(tst_m)), 0, None)
        q_store[q]["promise"].append(pq.mean()); q_store[q]["late"].append((y > pq).mean()); q_store[q]["n"].append(len(y))
        cols[f"pred_q{int(q * 100)}"] = pq
    q_frames.append(pd.DataFrame(cols))
q_roll = pd.concat(q_frames, ignore_index=True)

def wavg(v, n): return float(np.average(v, weights=n))
sim = pd.DataFrame([{"promesse": f"Quantile {q}", "durée promise moyenne (j)": wavg(s["promise"], s["n"]),
                     "taux de retard (%)": wavg(s["late"], s["n"]) * 100} for q, s in q_store.items()] +
                   [{"promesse": "Promesse actuelle d'Olist", "durée promise moyenne (j)": wavg(o_store["promise"], o_store["n"]),
                     "taux de retard (%)": wavg(o_store["late"], o_store["n"]) * 100}])
display(sim)

months_used = [m for m in roll_months if m in set(roll["month"])]
monthly = pd.DataFrame({"mois": months_used, "Olist : taux de retard (%)": np.array(o_store["late"]) * 100,
                        "Olist : promesse (j)": o_store["promise"],
                        **{f"q{int(q*100)} : taux de retard (%)": np.array(s["late"]) * 100 for q, s in q_store.items()},
                        **{f"q{int(q*100)} : promesse (j)": s["promise"] for q, s in q_store.items()}}).set_index("mois")
display(monthly)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].plot(sim["durée promise moyenne (j)"][:-1], sim["taux de retard (%)"][:-1], "o-", label="Promesse = quantile prédit")
for _, r in sim.iloc[:-1].iterrows():
    ax[0].annotate(r["promesse"], (r["durée promise moyenne (j)"], r["taux de retard (%)"]), textcoords="offset points", xytext=(6, 6))
ax[0].scatter(sim["durée promise moyenne (j)"].iloc[-1], sim["taux de retard (%)"].iloc[-1], c="red", s=90, zorder=5, label="Olist")
ax[0].set_xlabel("Durée promise moyenne (j)"); ax[0].set_ylabel("Taux de retard (%)"); ax[0].set_title("Compromis promesse / retard (évaluation glissante)"); ax[0].legend()
monthly[[c for c in monthly.columns if "taux de retard" in c]].plot(ax=ax[1], marker="o"); ax[1].set_title("Taux de retard par mois"); ax[1].set_ylabel("%")
plt.tight_layout(); plt.show()

## 9. Modèle final et fichiers de déploiement
Le modèle de production est entraîné sur **toutes** les commandes disponibles (pondération par la récence si la variante le prévoit). Fichiers écrits : modèle ponctuel, modèles quantiles, niveaux des catégories, liste des variables, prédictions glissantes par commande, tableaux de résultats et manifeste.

In [ ]:
now = full["order_purchase_timestamp"].max() + pd.Timedelta(days=1)
m_final, conv_final = fit_reg(full, chosen["feats"], chosen["weight"], now)
m_final.booster_.save_model(os.path.join(OUT_DIR, "model1_point_lgbm.txt"))
for q in Q_FINAL:
    mq, _ = fit_reg(full, chosen["feats"], chosen["weight"], now, "quantile", q)
    mq.booster_.save_model(os.path.join(OUT_DIR, f"model1_quantile_{int(round(q * 100)):02d}_lgbm.txt"))
json.dump(conv_final.levels, open(os.path.join(OUT_DIR, "model1_category_levels.json"), "w"), ensure_ascii=False, indent=1)
json.dump({"features": chosen["feats"], "categorical": [c for c in CATS if c in chosen["feats"]], "variant": final_variant,
           "quantiles": Q_FINAL}, open(os.path.join(OUT_DIR, "model1_features.json"), "w"), ensure_ascii=False, indent=1)

# Prédictions glissantes par commande (modèle retenu, baselines et quantiles)
keep = ["order_id", "month", "split", "delivery_days", "promised_days", f"pred::{final_variant}", "pred::Médiane par état (3 derniers mois)"]
out_pred = roll[keep].rename(columns={f"pred::{final_variant}": "pred_days", "pred::Médiane par état (3 derniers mois)": "baseline_state_recent"})
out_pred = out_pred.merge(q_roll, on="order_id", how="left")
out_pred.to_csv(os.path.join(OUT_DIR, "model1_rolling_predictions.csv"), index=False)
summary.to_csv(os.path.join(OUT_DIR, "model1_rolling_summary.csv"), index=False)
by_month.to_csv(os.path.join(OUT_DIR, "model1_mae_by_month.csv"))
static_test.to_csv(os.path.join(OUT_DIR, "model1_static_test_fair.csv"))
sim.to_csv(os.path.join(OUT_DIR, "model1_promise_simulator.csv"), index=False)
monthly.to_csv(os.path.join(OUT_DIR, "model1_promise_simulator_monthly.csv"))

manifest = {"final_variant": final_variant, "trained_on_rows": int(len(full)), "training_end": str(full["order_purchase_timestamp"].max()),
            "rolling_mae": float(fv["MAE (tous les mois)"]), "best_baseline": best_base["méthode"],
            "best_baseline_mae": float(best_base["MAE (tous les mois)"]), "olist_promise_mae": float(promise_mae),
            "recency_weighting": bool(chosen["weight"]), "half_life_days": HALF_LIFE if chosen["weight"] else None}
json.dump(manifest, open(os.path.join(OUT_DIR, "model1_manifest.json"), "w"), ensure_ascii=False, indent=2)
print("Fichiers de déploiement :", sorted(os.listdir(OUT_DIR)))

## Synthèse à rédiger
- **Gain équitable** du modèle contre les baselines (cellules 5 à 7), notamment contre la médiane par état des 3 derniers mois.
- **Effet des proxys du temps** : comparaison des variantes A et B (cellule 6).
- **Effet de la pondération par la récence** : variantes D et E contre B et A.
- **Simulateur de promesse** : durée promise et taux de retard mesurés mois par mois (cellule 8).
- **Décision** : variante retenue et fichiers à transmettre au déploiement (cellule 9).